# Procesamiento del lenguaje natural con modelos preentrenados

En la práctica anterior utilizamos bibliotecas como NLTK y TextBlob para procesar texto. Son eficientes y fáciles de usar, pero no siempre son las más adecuadas para idiomas distintos del inglés.

Los modelos preentrenados se han entrenado con grandes cantidades de datos y pueden procesar textos en distintos idiomas. Su entrenamiento requiere más datos y capacidad de cálculo; una vez entrenados, pueden ser eficientes. Además, reducen el trabajo manual y facilitan el procesamiento multilingüe.

En esta práctica usaremos modelos preentrenados para procesar texto en varios idiomas. Trabajaremos con `spaCy`, aunque existen otras colecciones, como `flair`.

## Preparación del entorno

Primero instalaremos `spaCy` y un modelo preentrenado para catalán: `ca_core_news_sm`.

In [88]:
!pip install spacy


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import spacy

spacy.cli.download("ca_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 19.0 MB/s eta 0:00:0000:0100:01



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


A continuación, cargaremos el modelo.

In [20]:
nlp = spacy.load("ca_core_news_sm")

## Tokenización

En spaCy, para procesar un texto basta con llamar al modelo `nlp`. Este devuelve un objeto `Doc`, que utilizaremos en muchas de las operaciones.

Podemos recorrer el objeto `Doc` como si fuera una lista para acceder a sus tokens.

In [94]:
doc = nlp("Els dilluns a la tarda són molt llargs.")
print([token.text for token in doc])

['Els', 'dilluns', 'a', 'la', 'tarda', 'són', 'molt', 'llargs', '.']


spaCy también permite obtener la categoría gramatical de cada palabra. Para ello accederemos al atributo `pos_` de cada token.

In [93]:
for token in doc:
    print(token.text, token.pos_)

Els DET
dilluns NOUN
a ADP
la DET
tarda NOUN
són AUX
molt ADV
llargs ADJ
. PUNCT
esquerra NOUN


Puedes consultar las categorías gramaticales en [Universal Dependencies](https://universaldependencies.org/u/pos/) o solicitar una descripción con `spacy.explain`.

In [40]:
for token in doc:
    print(token.text, token.pos_, spacy.explain(token.pos_))

El DET determiner
Barça PROPN proper noun
és AUX auxiliary
el DET determiner
millor ADJ adjective
equip NOUN noun
d ADP adposition
el DET determiner
món NOUN noun
. PUNCT punctuation
De ADP adposition
vegades NOUN noun
. PUNCT punctuation


### Tokenización de oraciones

A veces también necesitamos dividir el texto en oraciones. Para ello utilizaremos la propiedad `sents` del objeto `Doc`.

In [30]:
# Tokenitzem el text en frases

text = "El Barça és el millor equip del món. De vegades."
doc = nlp(text)

for sent in doc.sents:
    print(sent.text)

El Barça és el millor equip del món.
De vegades.


## Palabras vacías y signos de puntuación

Las *stopwords* o palabras vacías aportan poca información para ciertas tareas, como la clasificación de textos. Por eso, a menudo conviene eliminarlas antes del procesamiento.

En spaCy podemos identificar las palabras vacías mediante la propiedad `is_stop` de cada token del objeto `Doc`.

A continuación veremos cómo quitar las palabras vacías de un texto.

In [34]:
# Eliminem les stopwords d'un text

text = "El Barça és el millor equip del món. De vegades."
doc = nlp(text)
tokens = [token for token in doc if not token.is_stop]

tokens

[Barça, millor, equip, d, món, ., vegades, .]

También podemos eliminar los signos de puntuación junto con las palabras vacías.

In [36]:
tokens = [token for token in doc if not token.is_stop and not token.is_punct]
tokens

[Barça, millor, equip, d, món, vegades]

## Lematización y *stemming*

La lematización convierte una palabra a su forma base; por ejemplo, *está* pasa a *estar*. El *stemming* elimina afijos y puede reducir *está* a *est*.

Ambas técnicas reducen el vocabulario y la dimensionalidad de los vectores, lo que puede mejorar el rendimiento de los modelos.

spaCy no incluye *stemming*, pero sí lematización. Para obtener el lema de cada token utilizaremos el atributo `lemma_`. La lematización es más precisa y spaCy está orientado a aplicaciones de producción donde la precisión suele ser prioritaria.

In [46]:
# Lematitzem i stemitzem un text

text = "Series un bon cantant, si no fora per la veu."
doc = nlp(text)

# Lematitzem
lemmas = [token.lemma_ for token in doc]

lemmas

['seria',
 'un',
 'bo',
 'cantant',
 ',',
 'si',
 'no',
 'fora',
 'per',
 'el',
 'veu',
 '.']

## Representación del texto

Después de preprocesar el texto, debemos representarlo en un formato que pueda procesar el ordenador. spaCy está orientado a aplicaciones de producción e incluye métodos habituales, especialmente representaciones basadas en vectores de características.

### *Embeddings*

spaCy permite obtener los *embeddings* de documentos y palabras. Son vectores de características que representan el documento o la palabra y pueden utilizarse en tareas como clasificación y agrupación de textos.

Como estamos utilizando el modelo pequeño (`ca_core_news_sm`), sus vectores son menos precisos y están disponibles solo para documentos, no para palabras. Aun así, pueden servir para tareas sencillas. Para obtener vectores más precisos podemos utilizar los modelos mediano (`ca_core_news_md`) o grande (`ca_core_news_lg`).

In [47]:
doc.vector

array([ 1.7305475 , -0.12840228, -0.5396843 , -0.83717424, -1.3100742 ,
        0.17650956, -0.1620681 ,  0.48682687,  0.06764957,  0.18184917,
        0.8215378 , -0.40531054,  0.25735107,  0.93934184, -0.06517167,
        0.04939266, -0.4045429 ,  0.24763483,  1.0905582 ,  0.6970573 ,
       -0.31526443,  1.4649282 , -0.23816238, -0.4865251 , -0.09747415,
       -0.28767216,  0.5814107 , -0.7125825 ,  0.49832496,  0.8188111 ,
       -0.15080123,  0.1994148 ,  0.2723398 ,  0.8346703 , -0.7402689 ,
       -1.4915727 , -0.9905634 ,  0.42298186,  0.6075589 ,  0.29053253,
       -0.00331459, -1.3474356 , -0.58441734, -0.7741163 ,  0.05676066,
       -0.17020352,  0.40141544,  0.6308312 , -0.40938064,  0.6958816 ,
        0.3174107 ,  1.168798  , -0.04763202,  0.19581653,  0.08472364,
       -0.01598583,  0.00706673, -0.8146202 , -0.65326816, -0.6593003 ,
       -1.0186236 , -1.1242762 ,  0.1646251 ,  0.07394195, -0.7488492 ,
       -0.5852267 , -0.8377841 , -0.47709122,  0.6825135 ,  0.55

## Similitud entre textos

A partir de los *embeddings*, spaCy puede calcular la similitud entre dos documentos o palabras mediante la similitud del coseno entre sus vectores de características.

Esta función puede ser útil en sistemas de recomendación o buscadores de texto.

Para obtener vectores de palabras y comparar su similitud, necesitaremos cargar el modelo mediano (`ca_core_news_md`) o el grande (`ca_core_news_lg`).

### Ejemplo

In [65]:
# Carreguem un model de paraules

spacy.cli.download("ca_core_news_md")
nlp = spacy.load("ca_core_news_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 MB 11.7 MB/s eta 0:00:00a 0:00:01



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [67]:
# Creem dues frases i calculem la similitud

text1 = "M'agrada el futbol."

text2 = "M'agrada el bàsquet."

doc1 = nlp(text1)
doc2 = nlp(text2)

doc1.similarity(doc2)

0.9890047051139093

### Sinónimos

Podríamos utilizar la similitud para buscar sinónimos: si dos palabras superan un umbral de similitud, podríamos considerarlas sinónimas.

Sin embargo, esta técnica no es perfecta. Por ejemplo, *coche* y *vehículo* son sinónimos, pero su puntuación de similitud quizá no sea suficientemente alta. En esos casos podemos recurrir a un tesauro como WordNet. Utilizaremos `spacy-wordnet` para integrar WordNet con spaCy.

In [78]:
# Utilitzem wordnet i omw per obtenir sinònims

!pip install nltk


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [85]:
# Descarreguem els recursos necessaris

import nltk

nltk.download("wordnet")
nltk.download("omw")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /home/carles/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw to /home/carles/nltk_data...
[nltk_data]   Package omw is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/carles/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [87]:
# Obtenim els sinònims de la paraula 'gos'

from nltk.corpus import wordnet as wn

for synset in wn.synsets(u'gos',lang=('cat')):
    for lemma in synset.lemma_names(u'cat'):
        print(lemma)

ca
canis_familiaris
gos
gos_domèstic


# Conclusiones

En esta práctica hemos visto cómo preprocesar texto, representarlo para que lo procese el ordenador y utilizar algunas funciones de spaCy y bibliotecas relacionadas.

spaCy es una biblioteca potente y fácil de utilizar para procesar texto en distintos idiomas, con numerosas funciones útiles para tareas de procesamiento del lenguaje natural.